Nếu chạy trên Google Colab, upload file này lên Google Colab và chạy dòng dưới đây. Sau khi xong, chạy lệnh "pwd" để xem đã vào đúng directory Transformer-NLP-UET hay chưa.

Nếu chạy trên máy, hãy bỏ qua nó.

In [1]:
!git clone -b complete https://github.com/BobInVietnam/Transformer-NLP-UET.git
%cd /content/Transformer-NLP-UET

Cloning into 'Transformer-NLP-UET'...
remote: Enumerating objects: 127, done.
remote: Counting objects: 100% (127/127), done.
remote: Compressing objects: 100% (103/103), done.
remote: Total 127 (delta 45), reused 104 (delta 22), pack-reused 0 (from 0)
Receiving objects: 100% (127/127), 18.04 MiB | 29.18 MiB/s, done.
Resolving deltas: 100% (45/45), done.
/content/Transformer-NLP-UET


In [2]:
import pandas as pd
from transformer.transformer import Transformer

import os
import torch
import torch.nn as nn
from tqdm import tqdm
from torch.utils.data import DataLoader

from data.vocab import Vocabulary
from data.dataset import SummaryDataset
from data.dataloader import collate_fn

# Load dữ liệu từ dataset

In [3]:
train_df = pd.read_parquet("dataset/train-00000-of-00001.parquet")
valid_df = pd.read_parquet("dataset/valid-00000-of-00001.parquet")
print(train_df)


                                                 article  \
0      Gần 20 sự kiện được tổ chức trên toàn thành ph...   
1      Được thành lập năm 1897 tại Đức, Kempinski Hot...   
2      Ngoài di chuyển đến Tuần Châu bằng đường bộ, m...   
3      Với những tín đồ Phật giáo, bức tượng Phật ngọ...   
4      Số liệu của Tổng cục Thống kê công bố sáng 29/...   
...                                                  ...   
10770  Tình hình an ninh tại một số bang miền bắc Mya...   
10771  Tòa Hình sự Thượng thẩm quận Suwon hôm nay tuy...   
10772  Caolan Gormley, 26 tuổi, chủ công ty vận tải H...   
10773  Trong bối cảnh tình hình Myanmar đang xảy ra g...   
10774  Bộ Công an cho biết tính đến ngày 6/12, Cục Qu...   

                                                 summary  
0      Hà Nội tổ chức gần 20 sự kiện từ 19/4 đến 10/5...  
1      Kempinski Hotels là một thương hiệu nổi tiếng ...  
2      Bài viết giới thiệu các hoạt động vui chơi giả...  
3      Bức tượng Phật ngọc hòa bình thế giớ

In [4]:
print(train_df.columns)

Index(['article', 'summary'], dtype='object')


# Thông số model
Block code dưới đây nhằm load thông số model qua *model_config*, bao gồm các thông số về **số lượng từ vựng**, **độ lớn vector embedding**, **số lớp của model**, **số attention head**, **độ lớn lớp feed-forward**

*checkpoint_path* chỉ vào 1 file .pt (file chứa các thông số của model bao gồm optimizer để tiếp tục train). File này sẽ được xuất ra sau mỗi epoch ở phần training bên dưới.

Sau khi chỉnh sửa  thông số, chạy tất cả các block bên dưới bắt đầu từ block này.

In [8]:
start_epoch = 0
state_dict_to_load = None
optimizer_state_to_load = None
model_config = {
    "src_vocab_size": 40000,
    "tgt_vocab_size": 40000,
    "d_model": 512,
    "num_layers": 6,
    "num_heads": 8,
    "d_ff": 2048
}

BATCH_SIZE = 16

checkpoint_path = "transformer_checkpoint.pt"

In [9]:

if os.path.exists(checkpoint_path):
    print(f"Loading checkpoint from: {checkpoint_path}...")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    checkpoint = torch.load(checkpoint_path, map_location=device)

    # Extract historical tracking metadata
    start_epoch = checkpoint["epoch"]
    model_config = checkpoint["config"]
    state_dict_to_load = checkpoint["model_state_dict"]
    optimizer_state_to_load = checkpoint["optimizer_state_dict"]
    print(f"Checkpoint loaded successfully. Resuming from Epoch {start_epoch + 1}")
else:
    print("No file found yet")

Loading checkpoint from: transformer_checkpoint.pt...
Checkpoint loaded successfully. Resuming from Epoch 13


In [10]:
print(model_config)
print(start_epoch)

{'src_vocab_size': 40000, 'tgt_vocab_size': 40000, 'd_model': 512, 'num_layers': 6, 'num_heads': 8, 'd_ff': 2048}
12


Phần code load dữ liệu vào lớp vocab nhằm indexing.

In [11]:
train_input_vocab = Vocabulary(max_vocab=model_config["src_vocab_size"])
train_input_vocab.build_vocab(train_df["article"])

train_output_vocab = Vocabulary(max_vocab=model_config["tgt_vocab_size"])
train_output_vocab.build_vocab(train_df["summary"])

In [12]:
print(len(train_output_vocab.stoi))

40000


In [13]:
from data.dataloader import get_dataloader
train_dataloader = get_dataloader(dataframe=train_df, src_vocab=train_input_vocab, tgt_vocab=train_output_vocab, batch_size=BATCH_SIZE)
val_dataloader = get_dataloader(dataframe=valid_df, src_vocab=train_input_vocab, tgt_vocab=train_output_vocab, batch_size=BATCH_SIZE)

# Training the base Transformer

In [14]:

def train_epoch(model, dataloader, optimizer, criterion, device):
    """
    Trains the Transformer model for a single epoch.

    Args:
        model: The top-level unified Transformer module.
        dataloader: The train_dataloader fetching padded batch dictionaries.
        optimizer: The optimizer (e.g., torch.optim.Adam).
        criterion: The loss function (e.g., nn.CrossEntropyLoss).
        device: The compute hardware target ('cuda' or 'cpu').

    Returns:
        float: The average loss for this training epoch.
    """
    # 1. Put the model in training mode (activates Dropout layers)
    model.train()

    total_loss = 0.0

    # 2. Wrap your dataloader with tqdm to build the terminal progress bar
    # 'desc' adds a prefix label; 'leave=True' keeps the progress bar visible after completion
    progress_bar = tqdm(dataloader, desc="Training Epoch", leave=True)

    for batch_idx, batch in enumerate(progress_bar):
        # 3. Extract the named tensors from your custom collate_fn output dictionary
        input_ids = batch["input_ids"].to(device)                   # Source tokens
        attention_mask = batch["attention_mask"].unsqueeze(1).unsqueeze(2).to(device)           # Encoder padding mask
        decoder_input_ids = batch["decoder_input_ids"].to(device)   # Right-shifted target tokens
        labels = batch["labels"].to(device)                         # Expected output values

        # 4. Standard PyTorch optimization reset step
        optimizer.zero_grad()

        logits = model(src=input_ids, tgt=decoder_input_ids, src_mask=attention_mask)

        vocab_size = logits.size(-1)
        flat_logits = logits.view(-1, vocab_size)
        flat_labels = labels.view(-1)

        loss = criterion(flat_logits, flat_labels)

        loss.backward()
        optimizer.step()

        current_loss = loss.item()
        total_loss += current_loss
        running_avg_loss = total_loss / (batch_idx + 1)

        progress_bar.set_postfix({
            "batch_loss": f"{current_loss:.4f}",
            "avg_loss": f"{running_avg_loss:.4f}"
        })

    return total_loss / len(dataloader)

In [15]:
def validate_epoch(model, dataloader, criterion, device):
    """
    Evaluates the model on a validation dataset using Teacher Forcing
    to calculate the validation loss.
    """
    # 1. Put the model in evaluation mode (turns off Dropout)
    model.eval()

    total_loss = 0.0

    # 2. Wrap dataloader with tqdm progress tracking
    progress_bar = tqdm(dataloader, desc="Validation Epoch", leave=True)

    # 3. Disable gradient calculation to save memory and compute speed
    with torch.no_grad():
        for batch_idx, batch in enumerate(progress_bar):
            # Extract variables from your custom collate_fn dictionary
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].unsqueeze(1).unsqueeze(2).to(device)
            decoder_input_ids = batch["decoder_input_ids"].to(device)
            labels = batch["labels"].to(device)

            # Forward pass through the Transformer
            logits = model(src=input_ids, tgt=decoder_input_ids, src_mask=attention_mask)

            # Flatten shapes for CrossEntropy Loss
            vocab_size = logits.size(-1)
            flat_logits = logits.view(-1, vocab_size)
            flat_labels = labels.view(-1)

            # Calculate loss (ignores padding index 0 if configured in criterion)
            loss = criterion(flat_logits, flat_labels)

            total_loss += loss.item()
            running_avg_loss = total_loss / (batch_idx + 1)

            # Update the tqdm progress bar string on-the-fly
            progress_bar.set_postfix({
                "val_batch_loss": f"{loss.item():.4f}",
                "val_avg_loss": f"{running_avg_loss:.4f}"
            })

    return total_loss / len(dataloader)

In [16]:
LOG_FILE_PATH = "training_log.txt"
def log_to_file(message: str, file_path: str = LOG_FILE_PATH):
    """Utility helper to print a message to the console and append it to a log file."""
    print(message)  # Keeps the console terminal output visible
    with open(file_path, "a", encoding="utf-8") as f:
        f.write(message + "\n")

In [17]:
# 1. Choose your device hardware target
num_epochs = 20

if start_epoch == 0:
    with open(LOG_FILE_PATH, "w", encoding="utf-8") as f:
        f.write("=== Transformer Training Initialization ===\n")
        f.write(f"Model Configuration: {model_config}\n\n")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
log_to_file(f"Executing pipeline on device: {device}")

print(model_config)

transformer_model = Transformer(
    src_vocab_size=model_config["src_vocab_size"],
    tgt_vocab_size=model_config["tgt_vocab_size"],
    d_model=model_config["d_model"],
    num_heads=model_config["num_heads"],
    num_layers=model_config["num_layers"],
    d_ff=model_config["d_ff"]
)

if state_dict_to_load is not None:
    transformer_model.load_state_dict(state_dict_to_load)

transformer_model.to(device)

# 3. Setup Loss criterion
criterion = nn.CrossEntropyLoss(ignore_index=0)

# 4. Setup Optimizer (Matching the original paper setup parameters)
optimizer = torch.optim.Adam(transformer_model.parameters(), lr=1e-4, betas=(0.9, 0.98), eps=1e-9)

if optimizer_state_to_load is not None:
    optimizer.load_state_dict(optimizer_state_to_load)

# 5. Execute your complete epoch execution loop
for epoch in range(start_epoch, num_epochs):
    print(f"\n--- Epoch {epoch + 1}/{num_epochs} ---")

    # Run a training pass across the dataset via your dataloader
    epoch_loss = train_epoch(
        model=transformer_model,
        dataloader=train_dataloader,
        optimizer=optimizer,
        criterion=criterion,
        device=device
    )
    epoch_summary_str = f"Epoch {epoch + 1} Summary -> Overall Average Training Loss: {epoch_loss:.4f}"

    # Periodic evaluation log capture
    if (epoch + 1) % 2 == 0:
        val_loss = validate_epoch(
            model=transformer_model,
            dataloader=val_dataloader,
            criterion=criterion,
            device=device
        )
        # Append validation metrics onto our tracking string payload
        epoch_summary_str += f" | Validation Loss: {val_loss:.4f}"

    # Log the complete aggregated metrics directly into our file registry
    log_to_file(epoch_summary_str)

    checkpoint = {
        "epoch": epoch + 1,
        "model_state_dict": transformer_model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "loss": epoch_loss,
        "config": model_config  # <-- Injected directly here!
    }

    torch.save(checkpoint, "transformer_checkpoint.pt")
    log_to_file("Self-contained checkpoint saved cleanly.")


Executing pipeline on device: cpu
{'src_vocab_size': 40000, 'tgt_vocab_size': 40000, 'd_model': 512, 'num_layers': 6, 'num_heads': 8, 'd_ff': 2048}

--- Epoch 13/20 ---


Training Epoch:   0%|          | 0/674 [00:06<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
import torch
import torch.nn.functional as F

def generate_summary_beam_search(model, article_text, src_vocab, tgt_vocab, device, beam_size=3, max_len=50, alpha=0.6):
    """
    Generates a text summary from an input article using parallelized Beam Search decoding.

    Args:
        model: Your trained top-level unified Transformer model.
        article_text (str): The raw input source sentence/article string.
        src_vocab (Vocabulary): The source language Vocabulary helper instance.
        tgt_vocab (Vocabulary): The target language Vocabulary helper instance.
        device (torch.device): Active hardware compute target ('cuda' or 'cpu').
        beam_size (int): Number of alternative tracking paths to maintain (B).
        max_len (int): Maximum token generation threshold cutoff.
        alpha (float): Length normalization penalty coefficient.

    Returns:
        str: The final decoded summary sentence.
    """
    model.eval()

    # 1. Look up exact special token IDs dynamically from your Vocabulary maps
    pad_id = tgt_vocab.stoi["<PAD>"]
    unk_id = tgt_vocab.stoi["<UNK>"]
    sos_id = tgt_vocab.stoi["<SOS>"]
    eos_id = tgt_vocab.stoi["<EOS>"]

    # 2. Tokenize and numericalize the raw input article string using your helper
    src_tokens = article_text.lower().split()
    src_ids = src_vocab.numericalize(src_tokens)

    # 3. Convert to a 2D batch tensor of shape (1, src_seq_len) -> Batch Size = 1
    src_tensor = torch.tensor([src_ids], dtype=torch.long, device=device)

    # 4. Generate the 4D Encoder padding mask using your pad_id value
    src_mask = (src_tensor != pad_id).unsqueeze(1).unsqueeze(2).to(device)

    with torch.no_grad():
        # Pre-compute the Encoder context once to save substantial compute cycles
        encoder_output = model.encoder(x=src_tensor, mask=src_mask)

        # Initialize paths: list of tuples -> (cumulative_log_probability, token_ids_list)
        # We start with a single tracking point containing just the <SOS> token ID
        beams = [(0.0, [sos_id])]
        completed_beams = []

        # Autoregressive sequence generation loop
        for step in range(max_len):
            candidates = []

            # Package all current active tracking sequences into a unified parallel batch matrix
            beam_inputs = [tokens for _, tokens in beams]
            dec_input = torch.tensor(beam_inputs, dtype=torch.long, device=device)
            current_beam_size = dec_input.size(0)

            # Replicate encoder states and padding masks to match parallelized beam batching dimensions
            b_encoder_output = encoder_output.repeat(current_beam_size, 1, 1)
            b_src_mask = src_mask.repeat(current_beam_size, 1, 1, 1)

            # Compute Decoder forward pass
            logits = model.decoder(x=dec_input, encoder_output=b_encoder_output, src_mask=b_src_mask)

            # Isolate logits for the absolute LAST predicted token step sequence position
            next_token_logits = logits[:, -1, :]
            # Convert raw outputs into normalized log probabilities: log(P(x))
            log_probs = F.log_softmax(next_token_logits, dim=-1)

            # Snatch top B word choices for each path to minimize sorting load
            topk_log_probs, topk_ids = torch.topk(log_probs, beam_size, dim=-1)

            # Loop through active lanes to expand possible tracks
            for b_idx in range(current_beam_size):
                cum_log_prob, tokens = beams[b_idx]

                for k in range(beam_size):
                    next_token_prob = topk_log_probs[b_idx, k].item()
                    next_token_id = topk_ids[b_idx, k].item()

                    # Accumulate negative log probabilities: log(P(A)) + log(P(B))
                    new_cum_prob = cum_log_prob + next_token_prob
                    new_tokens = tokens + [next_token_id]

                    # Route path to completed container if it outputs an <EOS> boundary token
                    if next_token_id == eos_id:
                        completed_beams.append((new_cum_prob, new_tokens))
                    else:
                        candidates.append((new_cum_prob, new_tokens))

            # Early break check: if we have already gathered enough finished options, stop looping
            if len(completed_beams) >= beam_size:
                break

            # Globally sort all ongoing uncompleted routes and filter down to top B options
            candidates.sort(key=lambda x: x[0], reverse=True)
            beams = candidates[:beam_size]

            if not beams:
                break

        # Safety dump: if no route explicitly produced an <EOS> tag, dump open paths into the pool
        if not completed_beams:
            completed_beams = beams

        # 5. Length Normalization step
        # Shorter sequences accumulate fewer terms and naturally bias negative probabilities higher.
        # We counter this using the length normalization formula: Score = log_prob / (length^alpha)
        normalized_beams = []
        for log_prob, tokens in completed_beams:
            # Exclude the initial starting <SOS> token from length measurements
            length = len(tokens) - 1
            length = max(length, 1)   # Guard against divide-by-zero crashes
            norm_score = log_prob / (length ** alpha)
            normalized_beams.append((norm_score, tokens))

        # Sort and select the highest scoring track candidate
        normalized_beams.sort(key=lambda x: x[0], reverse=True)
        best_tokens = normalized_beams[0][1]

        # 6. Extraction and final text cleanup
        generated_ids = best_tokens[1:]  # Crop out the <SOS> token
        if eos_id in generated_ids:
            generated_ids = generated_ids[:generated_ids.index(eos_id)]  # Slice off <EOS> and everything after

        # 7. Use your Vocabulary.itos helper dictionary to map index positions back to string words
        generated_words = [tgt_vocab.itos.get(idx, "<UNK>") for idx in generated_ids]

        # Merge individual tokens into a final human-readable string sentence
        return " ".join(generated_words)

In [ ]:
sample_article = "Ghềnh đá Bàn Than nằm ở phía đông bắc xã đảo Tam Hải, huyện Núi Thành có dạng mặt bàn, nổi bật là những vách đá sắc đen như than, di tích còn lại của thềm biển cổ. Ghềnh dài 2 km, cao 40 m uốn quanh ngọn núi dọc bờ biển, nước trong xanh. Điểm đến này còn hoang sơ và không phí tham quan. Năm 2020, xã đảo Tam Hải được kênh truyền hìnhNational Geographic,Mỹ, điểm tên trong số 5 bãi biển đẹp nhất phía nam Việt Nam. Ghềnh đá Bàn Than nằm ở phía đông bắc xã đảo Tam Hải, huyện Núi Thành có dạng mặt bàn, nổi bật là những vách đá sắc đen như than, di tích còn lại của thềm biển cổ. Ghềnh dài 2 km, cao 40 m uốn quanh ngọn núi dọc bờ biển, nước trong xanh. Điểm đến này còn hoang sơ và không phí tham quan. Năm 2020, xã đảo Tam Hải được kênh truyền hìnhNational Geographic,Mỹ, điểm tên trong số 5 bãi biển đẹp nhất phía nam Việt Nam. Hai nữ du khách check in trên khối đá cao 5 m. Hai nữ du khách check in trên khối đá cao 5 m. Làng Thuận An yên bình với rặng dừa, bờ cát mịn, sóng nước dịu êm nằm cạnh ghềnh đá Bàn Than. Hiện nay du lịch ở xã đảo Tam Hải phát triển nhỏ lẻ, tự phát, chưa có một hoạt động xúc tiến, quy hoạch. Chính quyền huyện có chủ trương đầu tư 20 tỷ đồng để đóng 2 phà tải trọng 30 tấn đáp ứng vận chuyển người, phương tiện và mở rộng tuyến đường dẫn ra Bàn Than - Hòn Mang - Hòn Dứa. Huyện Núi Thành phối hợp với Sở Văn hóa Thể thao và Du lịch khảo sát và đưa xã đảo Tam Hải vào dự thảo Đề án các điểm định hướng phát triển du lịch cộng đồng của tỉnh. Làng Thuận An yên bình với rặng dừa, bờ cát mịn, sóng nước dịu êm nằm cạnh ghềnh đá Bàn Than. Hiện nay du lịch ở xã đảo Tam Hải phát triển nhỏ lẻ, tự phát, chưa có một hoạt động xúc tiến, quy hoạch. Chính quyền huyện có chủ trương đầu tư 20 tỷ đồng để đóng 2 phà tải trọng 30 tấn đáp ứng vận chuyển người, phương tiện và mở rộng tuyến đường dẫn ra Bàn Than - Hòn Mang - Hòn Dứa. Huyện Núi Thành phối hợp với Sở Văn hóa Thể thao và Du lịch khảo sát và đưa xã đảo Tam Hải vào dự thảo Đề án các điểm định hướng phát triển du lịch cộng đồng của tỉnh."

print("--- Executing Beam Search Verification ---")
print(f"Input text: {sample_article}")

# 2. Run generation pass
predicted_text = generate_summary_beam_search(
    model=transformer_model,
    article_text=sample_article,
    src_vocab=train_input_vocab,   # Source vocabulary instance
    tgt_vocab=train_output_vocab,  # Target vocabulary instance
    device=device,
    beam_size=4,                   # Track 4 alternate routes concurrently
    max_len=100,
    alpha=0.6                      # Length normalization penalty scale
)

print(f"Generated text output: {predicted_text}")

--- Executing Beam Search Verification ---
Input text: Ghềnh đá Bàn Than nằm ở phía đông bắc xã đảo Tam Hải, huyện Núi Thành có dạng mặt bàn, nổi bật là những vách đá sắc đen như than, di tích còn lại của thềm biển cổ. Ghềnh dài 2 km, cao 40 m uốn quanh ngọn núi dọc bờ biển, nước trong xanh. Điểm đến này còn hoang sơ và không phí tham quan. Năm 2020, xã đảo Tam Hải được kênh truyền hìnhNational Geographic,Mỹ, điểm tên trong số 5 bãi biển đẹp nhất phía nam Việt Nam. Ghềnh đá Bàn Than nằm ở phía đông bắc xã đảo Tam Hải, huyện Núi Thành có dạng mặt bàn, nổi bật là những vách đá sắc đen như than, di tích còn lại của thềm biển cổ. Ghềnh dài 2 km, cao 40 m uốn quanh ngọn núi dọc bờ biển, nước trong xanh. Điểm đến này còn hoang sơ và không phí tham quan. Năm 2020, xã đảo Tam Hải được kênh truyền hìnhNational Geographic,Mỹ, điểm tên trong số 5 bãi biển đẹp nhất phía nam Việt Nam. Hai nữ du khách check in trên khối đá cao 5 m. Hai nữ du khách check in trên khối đá cao 5 m. Làng Thuận An yên bìn